# Lesion Pre-Screening CNN - Example Usage

**⚠️ DISCLAIMER: This system is NOT for medical diagnosis!**

This notebook demonstrates how to use the lesion pre-screening system.


## Setup


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Import our modules
from src.model.cnn import create_model
from src.preprocessing.image_processing import ImagePreprocessor, LesionFeatureExtractor
from src.risk_assessment.scorer import LesionRiskScorer, calculate_abcde_score
from src.inference.predict import LesionPredictor

## 1. Model Architecture

Let's first examine the CNN architecture.

In [ ]:
# Create a model instance
model = create_model(model_type='cnn', num_classes=3)

# Print model summary
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Model architecture
print("\nModel Architecture:")
print(model)

## 2. Image Preprocessing

Demonstrate image preprocessing and augmentation.

In [ ]:
# Create preprocessor with augmentation
preprocessor = ImagePreprocessor(augment=True)

print("Preprocessing pipeline:")
print(preprocessor.transform)

## 3. Visual Feature Extraction

Extract ABCDE criteria features from a sample image.

In [ ]:
# Note: Replace with actual image path
# image_path = 'path/to/lesion/image.jpg'

# Example: Create a synthetic image for demonstration
demo_image = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)

# Extract features
extractor = LesionFeatureExtractor()
features = extractor.extract_all_features(demo_image)

print("Visual Features:")
for feature, value in features.items():
    print(f"  {feature}: {value:.3f}")

# Calculate ABCDE score
abcde = calculate_abcde_score(features)
print("\nABCDE Analysis:")
print(abcde)

## 4. Risk Assessment

Demonstrate risk scoring with sample predictions.

In [ ]:
# Create risk scorer
scorer = LesionRiskScorer(class_names=['benign', 'suspicious', 'malignant'])

# Simulate model predictions
predictions = np.array([0.2, 0.3, 0.5])  # benign, suspicious, malignant

# Sample visual features
visual_features = {
    'asymmetry': 0.7,
    'border_irregularity': 0.6,
    'color_variation': 0.8,
}

# Calculate comprehensive risk
risk_score = scorer.calculate_comprehensive_risk(predictions, visual_features)

print("Risk Assessment Results:")
print(f"Overall Score: {risk_score.overall_score:.1f}/100")
print(f"Risk Level: {risk_score.risk_level.value}")
print(f"Confidence: {risk_score.confidence:.1%}")
print("\nRecommendations:")
for rec in risk_score.recommendations:
    print(f"  • {rec}")

## 5. Visualization

Visualize predictions and risk factors.

In [ ]:
# Plot prediction probabilities
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Class probabilities
classes = ['Benign', 'Suspicious', 'Malignant']
colors = ['green', 'orange', 'red']
ax1.bar(classes, predictions, color=colors, alpha=0.7)
ax1.set_ylabel('Probability')
ax1.set_title('Model Predictions')
ax1.set_ylim([0, 1])
for i, v in enumerate(predictions):
    ax1.text(i, v + 0.02, f'{v:.1%}', ha='center')

# Visual features
feature_names = list(visual_features.keys())
feature_values = list(visual_features.values())
ax2.barh(feature_names, feature_values, color='steelblue', alpha=0.7)
ax2.set_xlabel('Score')
ax2.set_title('Visual Features (ABCDE)')
ax2.set_xlim([0, 1])
for i, v in enumerate(feature_values):
    ax2.text(v + 0.02, i, f'{v:.2f}', va='center')

plt.tight_layout()
plt.show()

## 6. End-to-End Inference

Complete workflow from image to PDF report.

**Note**: This requires a trained model.

In [ ]:
# Example (uncomment when you have a trained model)
"""
# Initialize predictor
predictor = LesionPredictor(
    model_path='../data/models/best_model.pth',
    class_names=['benign', 'suspicious', 'malignant'],
    device='cpu'  # or 'cuda'
)

# Process an image
risk_score = predictor.process_image(
    image_path='path/to/lesion.jpg',
    output_pdf='../outputs/report.pdf',
    patient_id='DEMO-001',
    verbose=True
)

print(f"\nReport generated!")
print(f"Overall Risk: {risk_score.overall_score:.1f}/100")
"""

print("See GETTING_STARTED.md for instructions on training a model first.")

## 7. Batch Processing

Process multiple images at once.

In [ ]:
# Example batch processing (uncomment when ready)
"""
import os
from pathlib import Path

# List of images to process
image_dir = Path('../data/test_images')
image_files = list(image_dir.glob('*.jpg'))

results = []

for img_path in image_files:
    print(f"Processing {img_path.name}...")
    
    risk_score = predictor.process_image(
        image_path=str(img_path),
        output_pdf=f'../outputs/{img_path.stem}_report.pdf',
        verbose=False
    )
    
    results.append({
        'filename': img_path.name,
        'risk_score': risk_score.overall_score,
        'risk_level': risk_score.risk_level.value,
        'confidence': risk_score.confidence
    })

# Display results
import pandas as pd
df = pd.DataFrame(results)
print(df)
"""

print("Batch processing example ready to use.")

## Conclusion

This notebook demonstrated:
1. Model architecture
2. Image preprocessing
3. Feature extraction
4. Risk assessment
5. Visualization
6. End-to-end inference
7. Batch processing

**Remember**: This system is for educational and pre-screening purposes only. Always consult medical professionals for diagnosis!
